<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/Langchain_Langsmith_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

langsmith_key = userdata.get('langsmith')

# 'LANGSMITH_TRACING' 대신 아래와 같이 'LANGCHAIN_TRACING_V2'로 수정
os.environ["LANGCHAIN_TRACING_V2"] = "true"  # <--- 이 부분
os.environ["LANGCHAIN_API_KEY"] = langsmith_key
os.environ["LANGCHAIN_PROJECT"] = "Colab-Test-Project"

In [2]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

In [3]:
from langchain_core.tools import tool

# 함수의 이름, docstring(설명), 매개변수 힌트가 모두 AI 모델의 프롬프트로 전달
@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱한 결과를 반환합니다."""
    return a * b

# 도구의 메타데이터 확인
print(multiply.name)         # 'multiply'
print(multiply.description)  # '두 정수를 곱한 결과를 반환합니다.'
print(multiply.args)         # {'a': {'title': 'A', 'type': 'integer'}, 'b': ...}

multiply
두 정수를 곱한 결과를 반환합니다.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [4]:
import os
from google.colab import userdata

YOUR_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

In [5]:
!pip install -q langchain-google-genai

from langchain.chat_models import init_chat_model

# Colab Secret에 GOOGLE_API_KEY를 등록해 두고 환경 변수로 설정합니다.
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception as e:
    print("경고: Colab 보안 비밀번호에서 'GOOGLE_API_KEY'를 가져오지 못했습니다.")
    print("API 키가 없다면 코드에 직접 지정하거나 Colab 비밀번호 설정을 확인하세요.\n")

model = init_chat_model(
    "gemini-3.5-flash",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [7]:
from langchain.agents import create_agent
# from deepagents import create_deep_agent

# SYSTEM_PROMPT 변수를 정의
SYSTEM_PROMPT = "You are a helpful assistant. Use your tools to search for and verify accurate information. Answer in a short and concise sentence"

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# deep_agent = create_deep_agent(
#     model=model,
#     tools=[fetch_text_from_url],
#     system_prompt=SYSTEM_PROMPT,
#     checkpointer=checkpointer,
# )

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

print("Running create_agent...", flush=True)
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
# print("Running create_deep_agent...", flush=True)
# deep_agent_result = deep_agent.invoke(
#     {"messages": [{"role": "user", "content": content}]},
#     config={"configurable": {"thread_id": "great-gatsby-da"}},
# )
print("\ncreate_agent:")
print(agent_result["messages"][-1].content_blocks)
# print("\ncreate_deep_agent:")
# print(deep_agent_result["messages"][-1].content_blocks)

Running create_agent...


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# agent_result의 최종 답변 텍스트만 출력합니다.
final_message = agent_result["messages"][-1]
print(final_message.content)

In [ ]:
from IPython.display import display, Markdown

raw_content = final_message.content

# 만약 content가 리스트 형태(블록 구조)라면 텍스트 블록만 합쳐줍니다.
if isinstance(raw_content, list):
    raw_text = "".join([block["text"] for block in raw_content if isinstance(block, dict) and block.get("type") == "text"])
elif isinstance(raw_content, str):
    raw_text = raw_content
else:
    raw_text = str(raw_content)

# 마크다운 형태로 예쁘게 출력합니다.
display(Markdown(raw_text))

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Gemini 모델 설정 (원하는 모델명을 기입합니다)
# 기본적으로 환경 변수에 설정된 GOOGLE_API_KEY를 자동으로 인식합니다.
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

# 2. 인터페이스와 메서드는 OpenAI 때와 완벽히 동일합니다.
response = model.invoke("The sky is")
print(response.content)


[{'type': 'text', 'text': "... **blue** (most of the time!). \n\nHere are a few different ways to complete that thought, depending on how you look at it:\n\n* **Scientifically:** ...blue because of a phenomenon called **Rayleigh scattering**. Sunlight contains all the colors of the rainbow, but the gases in Earth's atmosphere scatter the shorter, blue wavelengths of light more than other colors, making the sky appear blue to our eyes.\n* **Idiomatically:** ...**the limit** (meaning there are no boundaries to what you can achieve).\n* **Poetically:** ...a canvas that changes colors with the sunset, stars, and weather.\n* **Technically:** ...not actually a physical surface, but an atmosphere made mostly of nitrogen and oxygen stretching into space.\n* **Literally right now:** Depending on where you are, it might be cloudy, dark, orange, or clear!\n\nHow were you going to finish the sentence?", 'extras': {'signature': 'EtUUCtIUAWkUfRPdtXBzzfwDakOIKJHs80fXnueodOOSag66AquAoLOAHXWcJPWHy+Gddf